In [0]:
import pandas as pd
import numpy as np
import random
from datetime import datetime

# 1. Define the status options from your lookup table
status_data = {
    'Status_id': [1, 2, 3, 4, 5, 6],
    'Name': ['Instock', 'Low_Stock', 'Aging', 'Fast_Moving', 'Slow_Moving', 'Sold-Out'],
    'Description': ['Sufficient quantity available', 'Quantity below threshold', 'Sitting too long', 'High demand', 'Low demand', 'No Stock']
}
df_lookup = pd.DataFrame(status_data)

# 2. Configuration
years = range(2019, 2027)
divisions = ['Lifescience', 'MaterialScience', 'BatteryResearch']
skus = [f'LG{i:03}SR' for i in range(1, 41)]

all_inventory = []
product_catalog = {} 

# 3. Generation Logic
for year in years:
    # Determine number of SKUs (10% variance, range 10-40)
    if year == 2019:
        num_skus = 40
    else:
        num_skus = int(max(10, min(40, num_skus * random.uniform(0.9, 1.1))))
    
    current_batch_skus = random.sample(skus, num_skus)
    
    for sku in current_batch_skus:
        if sku not in product_catalog:
            product_catalog[sku] = {
                'division': random.choice(divisions),
                'base_price': random.uniform(100000, 2000000)
            }
        
        attr = product_catalog[sku]
        current_price = attr['base_price'] * (1.10 ** (year - 2019))
        
        # Pull status_id from the lookup table
        status_id = random.choice(df_lookup['Status_id'].tolist())
        
        record = {
            'invt_id': f"{year}_{sku}",
            'product_sku': sku,
            'quantity': random.randint(5, 50),
            'purchase_cost': round(current_price, 2),
            'warranty_date': datetime(year + 2, 4, 1),
            'division': attr['division'],
            'status_id': status_id
        }
        all_inventory.append(record)

df_inventory = pd.DataFrame(all_inventory)
display(df_inventory.head())

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS my_project_db;
USE my_project_db;
-- Save the DataFrame as a Delta Table in your workspace
-- This creates a 'bronze_inventory' table 


In [0]:
# Convert the Pandas DataFrame to a Spark DataFrame
spark_df = spark.createDataFrame(df_inventory)

# use the Spark DataFrame to write the table
spark_df.write.format("delta").mode("overwrite").saveAsTable("my_project_db.bronze_inventory")

print("Inventory data saved successfully to my_project_db.bronze_inventory!")

In [0]:
%sql
SELECT * FROM my_project_db.bronze_inventory;